# theoryminer — installation test in Google Colab

This notebook checks that `theoryminer` installs from GitHub with pip and that each public function works.
Run the cells from top to bottom. Each cell ends with a check. A check that fails stops the cell with an
`AssertionError`. At the end, save the notebook, so that the outputs are stored in the file.

## 1. Install from GitHub

In [ ]:
%pip install -q "theoryminer[umap] @ git+https://github.com/rasoulnorouzi/theoryminer.git"

## 2. The environment

In [ ]:
import os
import sys
import torch
import theoryminer

package_dir = os.path.dirname(theoryminer.__file__)
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("theoryminer:", package_dir)
print("data files :", sorted(os.listdir(os.path.join(package_dir, "data"))))

assert os.path.isfile(os.path.join(package_dir, "data", "ELSST_R5.rdf.gz")), "the ELSST file is missing from the package"
print("check passed: the package holds the ELSST file")

## 3. Download the sample papers

In [ ]:
import json
import urllib.request

REPO = "rasoulnorouzi/theoryminer"


def download_sample_files(folder="sample_files"):
    """Download every file of sample_files/ from GitHub into the folder."""
    listing_url = f"https://api.github.com/repos/{REPO}/contents/sample_files"
    with urllib.request.urlopen(listing_url) as response:
        listing = json.load(response)
    os.makedirs(folder, exist_ok=True)
    for item in listing:
        urllib.request.urlretrieve(item["download_url"], os.path.join(folder, item["name"]))
    return listing


listing = download_sample_files()
print(len(listing), "files:", sorted(item["name"] for item in listing))
assert len(listing) == 8, "expected 7 PDF files and SOURCES.md"
print("check passed: 8 files downloaded")

## 4. `harvest()`: one PDF, a folder, and bad files

In [ ]:
import shutil
from theoryminer import harvest

data = harvest("sample_files")
sent_ids = [s["sent_id"] for s in data["sentences"]]
doc_ids = {s["doc_id"] for s in data["sentences"]}

assert len(doc_ids) == 7, "expected 7 documents"
assert len(sent_ids) == len(set(sent_ids)), "sent_id is not unique"
print()
print("check passed:", len(sent_ids), "sentences from", len(doc_ids), "documents; every sent_id is unique")

In [ ]:
# A folder with one good PDF and three bad files.
os.makedirs("bad_folder", exist_ok=True)
shutil.copy("sample_files/barrech_2018_job_insecurity_health.pdf", "bad_folder/good.pdf")
with open("bad_folder/notes.txt", "w") as fh:
    fh.write("A text file, not a PDF.")
with open("bad_folder/webpage.pdf", "w") as fh:
    fh.write("<!DOCTYPE html><html><body>Access denied</body></html>")
with open("bad_folder/empty.pdf", "w") as fh:
    fh.write("")

mixed = harvest("bad_folder", cache=False)
skipped = {}
for row in mixed["dropped"]:
    if row["page"] == 0:
        skipped[row["doc_id"]] = row["drop_reason"]
print()
print("skipped files:", skipped)
assert skipped == {"notes": "not_pdf", "webpage": "unreadable_pdf", "empty": "unreadable_pdf"}
print("check passed: the bad files are logged, and the good file is read")

## 5. `causenet()`: find the cause-effect pairs

In [ ]:
from theoryminer import causenet

toy = causenet(["Smoking causes lung cancer and heart disease",
                "Exercise improves physical health and mental well-being",
                "This is a non-causal sentence about weather"])
print(toy[:2])
assert len(toy) >= 2, "the model found fewer relations than expected"
print("check passed: the model loads and finds relations")

In [ ]:
relations = causenet(data["sentences"])
rel_ids = [r["rel_id"] for r in relations]
assert len(rel_ids) == len(set(rel_ids)), "rel_id is not unique"
print()
print("check passed:", len(relations), "relations; every rel_id is unique")
for r in relations[:5]:
    print(f"  [{r['rel_id']} p{r['page']}] {r['cause']!r} -> {r['effect']!r}")

## 6. The harmonizer: ELSST, grouping, clustering, names

In [ ]:
from theoryminer.harmonizer import (
    standardize_constructs, standardize_groups, pairwise_grouping, clusterer, label_groups)

probe = ["loneliness", "social trust", "unemployment", "intrinsic motivation"]
matches = standardize_constructs(probe, top=3)
print()
for span in probe:
    best = matches[span][0]
    print(f"  {span:<22} -> {best['leaf']} ({best['score']:.2f})")
assert all(len(matches[span]) == 3 for span in probe)
print("check passed: the packaged ELSST file loads, and the spans get concepts")

In [ ]:
groups = pairwise_grouping(relations, threshold=0.69)
clusters = clusterer(relations)
clusters_umap = clusterer(relations, umap=dict(n_components=5, n_neighbors=15, min_dist=0.0))
named = standardize_groups(clusters, name_by="mean_sim", top=3)
labelled = label_groups(clusters, by="both", top=3)

spans = {r["cause"] for r in relations} | {r["effect"] for r in relations}
assert set(groups) == spans and set(clusters) == spans, "a span is missing from a result"
print()
print("check passed: every span has a group, a cluster and a name")

## 7. Summary

In [ ]:
print("sentences :", len(data["sentences"]))
print("relations :", len(relations))
print("spans     :", len(spans))
print("clusters  :", len({g["group"] for g in clusters.values() if g["group"] != -1}))
print("ALL CHECKS PASSED")